In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cu

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/BERT_IFND"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
import functools
import time
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# ─── 5. Preprocess với dataset.map() ─────────────────────────────────────────
model_name = "bert-base-uncased"
tokenizer  = BertTokenizer.from_pretrained(model_name)

def preprocess(example):
    inputs = tokenizer(
        str(example['text']),
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Model ─────────────────────────────────────────────────────────────────
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.2f}M")
print(f"Trainable params: {trainable_params/1e6:.2f}M")

# ─── 7. Metrics ───────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 8. Training Args ─────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

# ─── 9. Trainer ───────────────────────────────────────────────────────────────
torch.load = functools.partial(torch.load, weights_only=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

last_checkpoint = get_last_checkpoint(SAVE_DIR)
if last_checkpoint is not None:
    print(f"▶️ Resuming from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🆕 Training from scratch")
    trainer.train()

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("✅ Saved to:", SAVE_DIR)

# ─── 10. Latency + VRAM ───────────────────────────────────────────────────────
model.eval()
dummy_inputs = tokenizer(
    "This is a sample news headline for inference measurement.",
    truncation=True, padding="max_length",
    max_length=128, return_tensors="pt"
)
input_ids      = dummy_inputs["input_ids"].to(device)
attention_mask = dummy_inputs["attention_mask"].to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids=input_ids, attention_mask=attention_mask)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

print(f"\n{'='*40}")
print(f"Latency median: {np.median(latencies):.2f} ms")
print(f"VRAM (peak)   : {vram:.1f} MB")
print(f"{'='*40}")

# ─── 11. Test Evaluation ──────────────────────────────────────────────────────
print("\n📊 Evaluating on test set...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels, preds, average="binary")
print(f"\n{'='*40}")
print(f"Test Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
print(f"Test Precision: {precision*100:.2f}%")
print(f"Test Recall   : {recall*100:.2f}%")
print(f"Test F1       : {f1*100:.2f}%")
print(f"Test AUC      : {roc_auc_score(labels, probs[:, 1]):.5f}")
print(f"{'='*40}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

...

Total params    : 109.48M
Trainable params: 109.48M
🆕 Training from scratch


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.154600,0.045356,0.989544,0.991347,0.995037,0.993189,0.998542
2,0.025400,0.081497,0.986692,0.998741,0.983871,0.991250,0.998827
3,0.008800,0.040225,0.990494,0.990148,0.997519,0.993820,0.998719
4,0.007800,0.048558,0.990494,0.992574,0.995037,0.993804,0.998694
5,0.002800,0.054349,0.991445,0.993804,0.995037,0.994420,0.998792


✅ Saved to: /content/drive/MyDrive/BERT_IFND

Latency median: 12.60 ms
VRAM (peak)   : 1491.7 MB

📊 Evaluating on test set...



Test Accuracy : 99.43%
Test Precision: 99.38%
Test Recall   : 99.88%
Test F1       : 99.63%
Test AUC      : 0.99859
